# SuperStore

In [ ]:
import pandas as pd
import duckdb
import numpy as np

from modules.const import DUCKDB_FILE_PATH

with duckdb.connect(DUCKDB_FILE_PATH) as conn:
    facts     = conn.sql("SELECT * FROM FactSales").df()
    customers = conn.sql("SELECT * FROM DimCustomers").df()
    products  = conn.sql("SELECT * FROM DimProducts").df()
    geography = conn.sql("SELECT * FROM DimGeography").df()
    conn.sql("DESCRIBE FactSales").show()

facts.dtypes

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Row_ID      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ OrderID     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ OrderDate   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ShipDate    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ShipMode    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ProductKey  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ CustomerKey │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ GeoKey      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Sales       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



Row_ID                  int64
OrderID                   str
OrderDate      datetime64[us]
ShipDate       datetime64[us]
ShipMode                  str
ProductKey            float64
CustomerKey             int64
GeoKey                  int64
Sales                 float64
dtype: object

## 1. YoY Revenue growth per Category  

Revenue per anno e categoria,  
variazione percentuale anno su anno.

| Colonna        | Descrizione |
|---             | ---|
| `year`         | Anno dell'ordine |
| `category`     | Categoria prodotto |
| `revenue`      | Revenue totale dell'anno |
| `prev_revenue` | Revenue anno precedente |
| `yoy_pct`      | Variazione % su anno precedente |

In [56]:
(
    facts
    .assign(Year=facts['OrderDate'].dt.year)
    .merge(products[['ProductKey', 'Category']], on='ProductKey', how='inner')
    .groupby(['Year', 'Category'], as_index=False)
    .aggregate(Revenue=('Sales', 'sum'))
    .assign(
        Prev_Revenue=lambda x: x.groupby('Category')['Revenue'].shift(1),
        yoy_pct=lambda x: (x['Revenue'] / x['Prev_Revenue'] - 1) * 100
    )
    .sort_values(['Category', 'Year'])
    .round(2)
)

,Year,Category,Revenue,Prev_Revenue,yoy_pct
0,2015,Furniture,156181.07,NaN,NaN
3,2016,Furniture,162826.97,156181.07,4.26
6,2017,Furniture,194785.30,162826.97,19.63
9,2018,Furniture,211269.77,194785.30,8.46
1,2015,Office Supplies,148987.97,NaN,NaN
4,2016,Office Supplies,131934.73,148987.97,-11.45
7,2017,Office Supplies,180993.53,131934.73,37.18
10,2018,Office Supplies,237115.69,180993.53,31.01
2,2015,Technology,170773.23,NaN,NaN
5,2016,Technology,155365.90,170773.23,-9.02


## 2. Customer cohort retention  

Raggruppamento di clienti per anno del primo acquisto (cohort),  
e misura del revenue generato da quella cohort negli anni successivi. 

| Colonna                    | Descrizione |
|---                         | ---|
| `cohort_year`              | Anno del primo acquisto del cliente |
| `order_year`               | Anno in cui la cohort ha generato revenue |
| `periods_since_first`      | Distanza in anni tra `order_year` e `cohort_year` |
| `customers`                | Clienti distinti attivi in quell'anno |
| `revenue`                  | Revenue totale della cohort in quell'anno |
| `avg_revenue_per_customer` | Revenue medio per cliente attivo |

In [100]:
(
    facts
    .loc[:, ['CustomerKey', 'OrderDate', 'Sales']]
    .assign(
        Order_Year=lambda x: x['OrderDate'].dt.year,
        Cohort_Year=lambda x: x.groupby('CustomerKey')['OrderDate'].transform('min').dt.year,
        Periods_Since_First=lambda x: x['Order_Year'] - x['Cohort_Year'],
    )
    .groupby(['Cohort_Year', 'Order_Year', 'Periods_Since_First'], as_index=False)
    .aggregate(
        Customers=('CustomerKey', 'nunique'),
        Revenue=('Sales', 'sum')
    )
    .assign(Avg_Revenue_per_Customer=lambda x: x['Revenue'] / x['Customers'])
    .round(2)
)

,Cohort_Year,Order_Year,Periods_Since_First,Customers,Revenue,Avg_Revenue_per_Customer
0,2015,2015,0,589,479856.21,814.70
1,2015,2016,1,426,347204.34,815.03
2,2015,2017,2,476,445144.57,935.18
3,2015,2018,3,510,522227.45,1023.98
4,2016,2016,0,141,112231.66,795.97
5,2016,2017,1,107,100662.82,940.77
6,2016,2018,2,124,131199.83,1058.06
7,2017,2017,0,52,54385.17,1045.87
8,2017,2018,1,45,61112.93,1358.07
9,2018,2018,0,11,7511.80,682.89


## 3. ABC analysis sui prodotti  

Ordinamento dei prodotti per revenue decrescente,  
calcolo del revenue cumulato,  
classifica in A (primo 80%), B (80–95%), C (coda).

| Colonna          | Descrizione |
|---               | ---|
| `product_id`     | Identificativo prodotto |
| `product_name`   | Nome prodotto |
| `category`       | Categoria |
| `subcategory`    | Sottocategoria |
| `revenue`        | Revenue totale del prodotto |
| `revenue_pct`    | Percentuale sul revenue totale |
| `cumulative_pct` | Percentuale cumulata (ordinata per revenue desc) |
| `class`          | Classificazione: `A` (0–80%), `B` (80–95%), `C` (95–100%) |

In [ ]:
(
    products[['ProductKey', 'ProductID', 'ProductName', 'Category', 'SubCategory']]
    .merge(facts[['ProductKey', 'Sales']], on='ProductKey', how='inner')
    .groupby(['ProductID', 'ProductName', 'Category', 'SubCategory'], as_index=False)
    .aggregate(Revenue=('Sales', 'sum'))
    .assign(revenue_pct=lambda x: x['Revenue'] / facts['Sales'].sum() * 100)
    .sort_values('revenue_pct', ascending=False)
    .assign(
        cumulative_pct=lambda x: x['revenue_pct'].cumsum(),
        class_=lambda x: pd.cut(
            x['cumulative_pct'],
            bins=[-np.inf, 80, 95, 100],
            labels=['A', 'B', 'C']
        )
    )
    .round({'Revenue': 2, 'revenue_pct': 6, 'cumulative_pct': 6})
)

,ProductID,ProductName,Category,SubCategory,Revenue,revenue_pct,cumulative_pct,class_
1613,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,61599.82,2.723804,2.723804,A
776,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,27453.38,1.213926,3.937730,A
1641,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,22638.48,1.001022,4.938752,A
80,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,21870.58,0.967067,5.905819,A
691,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,19823.48,0.876549,6.782368,A
...,...,...,...,...,...,...,...,...
1445,OFF-SU-10003936,Acme Serrated Blade Letter Opener,Office Supplies,Supplies,7.63,0.000337,98.607473,C
862,OFF-EN-10001535,Grip Seal Envelopes,Office Supplies,Envelopes,7.07,0.000313,98.607785,C
1016,OFF-PA-10000048,Xerox 20,Office Supplies,Paper,6.48,0.000287,98.608072,C
988,OFF-LA-10003388,Avery 5,Office Supplies,Labels,5.76,0.000255,98.608327,C


## 4. Shipping performance  

Calcolo delay effettivo (`ShipDate` − `OrderDate`) per ogni ShipMode,  
confronto con il delay atteso (codificato in `modules.randomizer.Randomizer`, vd sotto),  
e isolamento ordini anomali oltre la soglia.

`Same Day`:       range(0, 2)  
`Standard Class`: range(4, 8)  
`First Class`:    range(1, 4)  
`Second Class`:   range(2, 6)  

| Colonna          | Descrizione |
|---               | ---|
| `ship_mode`      | Modalità di spedizione |
| `orders`         | Numero ordini |
| `avg_delay_days` | Delay medio effettivo in giorni (ShipDate − OrderDate) |
| `min_delay_days` | Delay minimo |
| `max_delay_days` | Delay massimo |
| `expected_min`   | Delay minimo atteso per quel ShipMode |
| `expected_max`   | Delay massimo atteso per quel ShipMode |
| `anomalies`      | Ordini fuori dalla finestra attesa |
| `anomaly_pct`    | Percentuale di ordini anomali |

In [192]:
expected_values = (
    facts[['ShipMode']]
    .drop_duplicates()
    .assign(
        Expected_Min=lambda x: x['ShipMode'].map({
            'Same Day': 0,
            'Standard Class': 4,
            'First Class': 1,
            'Second Class': 2
        }),
        Expected_Max=lambda x: x['ShipMode'].map({
            'Same Day': 1,
            'Standard Class': 7,
            'First Class': 3,
            'Second Class': 5
        }),
    )
)

anomalies = (
    facts[['ShipMode', 'ShipDate', 'OrderDate']]
    .assign(Delay_Days=lambda x: (x['ShipDate'] - x['OrderDate']).dt.days)
    .merge(expected_values, on='ShipMode', how='inner')
    .assign(Anomalies=lambda x: x['Delay_Days'].lt(x['Expected_Min']) | x['Delay_Days'].gt(x['Expected_Max']))
    .groupby('ShipMode', as_index=False)
    .aggregate(
        Anomalies=('Anomalies', 'sum'),
        Anomaly_pct=('Anomalies', 'mean')
    )
    .assign(Anomaly_pct=lambda x: (x['Anomaly_pct'] * 100).round(3))
)

(
    facts
    .assign(Delay_Days=lambda x: (x['ShipDate'] - x['OrderDate']).dt.days)
    .groupby('ShipMode', as_index=False)
    .aggregate(
        Avg_Delay_Days=('Delay_Days', 'mean'),
        Min_Delay_Days=('Delay_Days', 'min'),
        Max_Delay_Days=('Delay_Days', 'max')
    )
    .round(2)
    .merge(expected_values, on='ShipMode', how='inner')
    .merge(anomalies, on='ShipMode', how='inner')
    .sort_values('Anomaly_pct', ascending=False)
)

,ShipMode,Avg_Delay_Days,Min_Delay_Days,Max_Delay_Days,Expected_Min,Expected_Max,Anomalies,Anomaly_pct
0,First Class,2.18,1,4,1,3,1,0.067
2,Second Class,3.25,1,5,2,5,1,0.053
3,Standard Class,5.01,3,7,4,7,1,0.017
1,Same Day,0.04,0,1,0,1,0,0.000


## 5. Geographic market penetration  

Revenue e numero ordini per Stato,  
con in aggiunta il revenue medio per ordine  
e il rank nazionale.  
Identificare Stati ad alto volume ma basso ticket medio (potenziale upselling) e viceversa.  

| Colonna                | Descrizione |
|---                     | ---|
| `region`               | Regione |
| `state`                | Stato |
| `orders`               | Numero ordini distinti |
| `revenue`              | Revenue totale |
| `avg_order_value`      | Revenue medio per ordine |
| `national_revenue_pct` | Percentuale sul revenue nazionale |
| `revenue_rank`         | Rank per revenue all'interno della regione |
| `segment`              | `high_volume_low_ticket` / `low_volume_high_ticket` / `other` |

In [221]:
(
    facts
    .merge(geography, how='left', on='GeoKey')
    .groupby(['Region', 'State'], as_index=False)
    .aggregate(
        Orders=('OrderID', 'nunique'),
        Revenue=('Sales', 'sum'),
        Avg_Order_Value=('Sales', 'mean')
    )
    .assign(
        National_Revenue_Pct=lambda x: x['Revenue'] / facts['Sales'].sum() * 100,
        Revenue_Rank=lambda x: x.groupby(['Region'])['Revenue'].rank(method='min', ascending=False),
        Segment=lambda x: np.select(
            [x['Orders'].ge(x['Orders'].quantile(0.5)) & x['Revenue'].le(x['Revenue'].quantile(0.5)),
                x['Orders'].le(x['Orders'].quantile(0.5)) & x['Revenue'].ge(x['Revenue'].quantile(0.5))],
            ['High Volume Low Ticket',
                'Low Volume High Ticket'],
            default='Other'
        )
    )
    .round(2)
    .sort_values(['Region', 'Revenue_Rank'])
)

,Region,State,Orders,Revenue,Avg_Order_Value,National_Revenue_Pct,Revenue_Rank,Segment
11,Central,Texas,480,168572.53,173.25,7.45,1.0,Other
0,Central,Illinois,270,79236.52,164.05,3.50,2.0,Other
4,Central,Michigan,116,76136.07,300.93,3.37,3.0,Other
1,Central,Indiana,70,48718.40,360.88,2.15,4.0,Other
12,Central,Wisconsin,52,31173.43,296.89,1.38,5.0,Other
5,Central,Minnesota,44,29863.15,335.54,1.32,6.0,Other
6,Central,Missouri,30,22205.15,336.44,0.98,7.0,Low Volume High Ticket
9,Central,Oklahoma,34,19683.39,298.23,0.87,8.0,Other
7,Central,Nebraska,23,7464.93,196.45,0.33,9.0,Other
2,Central,Iowa,16,4443.56,170.91,0.20,10.0,Other
